# StressID and Experiment Dataset comparison

In [89]:
%matplotlib widget
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

STRESSID_DATA_PATH = "../../.."
STRESSID_LABELS_SEPARATOR = ","
STRESSID_LABELS_FILENAME = f"{STRESSID_DATA_PATH}/stressID/labels.csv"
STRESSID_FEATURES_SEPARATOR = ","
STRESSID_FEATURES_DIRECTORY = f"{STRESSID_DATA_PATH}/Reprod-Features-NK"
STRESSID_FEATURES_FILENAME = f"{STRESSID_FEATURES_DIRECTORY}/ecg_eda_features.csv"

EXPDATA_DATA_PATH = "../../../experiment-data"
EXPDATA_LABELS_SEPARATOR = ","
EXPDATA_LABELS_FILENAME = f"{EXPDATA_DATA_PATH}/labels.csv"
EXPDATA_FEATURES_SEPARATOR = ";"
EXPDATA_FEATURES_DIRECTORY = f"{EXPDATA_DATA_PATH}/extracted-features"
EXPDATA_FEATURES_FILENAME = f"{EXPDATA_FEATURES_DIRECTORY}/all_features.csv"

In [91]:
stressid_df = pd.read_csv(STRESSID_FEATURES_FILENAME, sep=STRESSID_FEATURES_SEPARATOR, index_col=0)
n_rows, n_cols = stressid_df.shape
print("======================================================")
print(f"StressID dataset contains {n_rows} samples with {n_cols} features")
# display(stressid_df)

si_labels_df = pd.read_csv(STRESSID_LABELS_FILENAME, sep=STRESSID_LABELS_SEPARATOR, index_col=0)
n_rows, n_cols = si_labels_df.shape
print("======================================================")
print(f"StressID labels contains {n_rows} labels with {n_cols} types of classifications")
# display(si_labels_df)

# Selecting rows that actually have entries both labels and samples
idx = list(stressid_df.merge(si_labels_df, left_index=True, right_index=True).index)
si_labels = si_labels_df.loc[idx]
si_X = stressid_df.loc[idx]
n_rows, n_cols = si_X.shape
print("======================================================")
print(f"StressID classification dataset contains {n_rows} samples with {n_cols} features")

si_bclass_labels = si_labels["binary-stress"]
# display(bclass_labels)


expdata_df = pd.read_csv(EXPDATA_FEATURES_FILENAME, sep=EXPDATA_FEATURES_SEPARATOR, index_col=0)
n_rows, n_cols = expdata_df.shape
print("\n\n======================================================")
print(f"ExpData dataset contains {n_rows} samples with {n_cols} features")
# display(expdata_df)

ed_labels_df = pd.read_csv(EXPDATA_LABELS_FILENAME, sep=EXPDATA_LABELS_SEPARATOR, index_col=0)
n_rows, n_cols = ed_labels_df.shape
print("======================================================")
print(f"ExpData labels contains {n_rows} labels with {n_cols} types of classifications")
# display(ed_labels_df)

# Selecting rows that actually have entries both labels and samples
idx = list(expdata_df.merge(ed_labels_df, left_index=True, right_index=True).index)
ed_labels = ed_labels_df.loc[idx]
ed_X = expdata_df.loc[idx]
n_rows, n_cols = ed_X.shape
print("======================================================")
print(f"ExpData classification dataset contains {n_rows} samples with {n_cols} features")

ed_bclass_labels = ed_labels["binary-stress"]
# display(bclass_labels)

StressID dataset contains 773 samples with 70 features
StressID labels contains 700 labels with 3 types of classifications
StressID classification dataset contains 699 samples with 70 features


ExpData dataset contains 147 samples with 70 features
ExpData labels contains 126 labels with 3 types of classifications
ExpData classification dataset contains 126 samples with 70 features


### Dimension reduction

In [94]:
# PCA
STD_EXPL_RATIO = 0.95

si_pca_95p = PCA(n_components=STD_EXPL_RATIO, svd_solver="full")
scaled = StandardScaler().fit_transform(si_X)
si_X_95p = si_pca_95p.fit_transform(scaled)

n_rows, n_cols = si_X_95p.shape
ratios = [(f"{x * 100:.2f}") for x in si_pca_95p.explained_variance_ratio_]
print(f"StressID classification dataset reduced to {n_cols} derived features that explains >{STD_EXPL_RATIO*100}% of variance in the dataset")
print(f"Features contribution ratio to variance: {ratios}")



ed_pca_95p = PCA(n_components=STD_EXPL_RATIO, svd_solver="full")
scaled = StandardScaler().fit_transform(ed_X)
ed_X_95p = ed_pca_95p.fit_transform(scaled)

n_rows, n_cols = ed_X_95p.shape
ratios = [(f"{x * 100:.2f}") for x in ed_pca_95p.explained_variance_ratio_]
print(
    f"\n\nExpData classification dataset reduced to {n_cols} derived features that explains >{STD_EXPL_RATIO * 100}% of variance in the dataset"
)
print(f"Features contribution ratio to variance: {ratios}")

StressID classification dataset reduced to 26 derived features that explains >95.0% of variance in the dataset
Features contribution ratio to variance: ['19.51', '12.45', '9.35', '7.55', '6.15', '5.42', '4.47', '3.83', '3.16', '2.71', '2.31', '1.97', '1.91', '1.78', '1.46', '1.41', '1.32', '1.28', '1.13', '1.03', '0.93', '0.89', '0.85', '0.78', '0.69', '0.66']


ExpData classification dataset reduced to 22 derived features that explains >95.0% of variance in the dataset
Features contribution ratio to variance: ['25.23', '12.55', '9.76', '7.94', '6.48', '5.07', '4.07', '3.48', '2.99', '2.63', '2.16', '2.00', '1.67', '1.54', '1.43', '1.33', '1.07', '1.01', '0.92', '0.81', '0.70', '0.64']


In [96]:
SPLITS = 10
RAN_STATE = 21

In [104]:
# RFECV for StressID
estimator = RandomForestClassifier(max_depth=5, random_state=RAN_STATE)
cv = StratifiedKFold(n_splits=SPLITS, shuffle=True, random_state=RAN_STATE)
selector = RFECV(estimator=estimator, step=2, cv=cv, scoring="balanced_accuracy")
selector.fit(si_X, si_bclass_labels)
features_mask = selector.support_
si_X_rfe = si_X.loc[:, features_mask]

features_scores = { "scores": [], "features": [] }
for score, feat in zip(selector.estimator_.feature_importances_, si_X_rfe.columns):
    features_scores["scores"].append(score)
    features_scores["features"].append(feat)
features_scores_df = pd.DataFrame(features_scores).sort_values(by="scores", ascending=False)
features_scores_df = features_scores_df.reset_index(drop=True)
features_scores_df.index = features_scores_df.index + 1
print("Selected features scores for StressID")
display(features_scores_df)

Selected features scores for StressID


,scores,features
1,0.126214,pNN20
2,0.107441,ULF
3,0.058393,sampEn
4,0.047258,pNN50
5,0.026279,apEn
6,0.022273,totalpower
7,0.020550,min_ecg
8,0.020404,min_scl
9,0.019703,sd_ecg
10,0.018357,median_eda


In [105]:
# RFECV for ExpData
estimator = RandomForestClassifier(max_depth=5, random_state=RAN_STATE)
cv = StratifiedKFold(n_splits=SPLITS, shuffle=True, random_state=RAN_STATE)
selector = RFECV(estimator=estimator, step=2, cv=cv, scoring="balanced_accuracy")
selector.fit(ed_X, ed_bclass_labels)
features_mask = selector.support_
ed_X_rfe = ed_X.loc[:, features_mask]

features_scores = {"scores": [], "features": []}
for score, feat in zip(selector.estimator_.feature_importances_, ed_X_rfe.columns):
    features_scores["scores"].append(score)
    features_scores["features"].append(feat)
features_scores_df = pd.DataFrame(features_scores).sort_values(by="scores", ascending=False)
features_scores_df = features_scores_df.reset_index(drop=True)
features_scores_df.index = features_scores_df.index + 1
print("Selected features scores for ExpData")
display(features_scores_df)

Selected features scores for ExpData


,scores,features
1,0.098942,pNN20
2,0.091173,sumAmpSCR
3,0.089739,SD1SD2
4,0.087833,median_ecg
5,0.081512,sd_scl
6,0.078022,sdHR
7,0.069198,SD1
8,0.068391,sk_eda
9,0.062715,scl_slope
10,0.062085,modeHR
